# Anomaly and Outlier Detection

**Anomaly Detection** is the process of identifying rare items, events, or observations that raise suspicion by differing significantly from the majority of the data. Applications range from credit card fraud detection and network intrusion monitoring to industrial equipment failure prediction and medical diagnosis.

In these detailed notes, we will cover:
1. **Core Terminology:** Outliers vs. Novelties, and Point, Contextual, and Collective anomalies.
2. **Classical Detection Techniques:** Statistical (Z-Score, IQR, Mahalanobis), Distance-based (KNN), density-based (LOF), tree-based (Isolation Forest), and kernel-based (One-Class SVM) algorithms.
3. **Isolation Forest & One-Class SVM Mechanics:** Detailed look at recursive partitioning and boundary separation.
4. **Practical Python Demos:**
   - **Demo A:** Custom scratch implementation of a statistical `ZScoreOutlierDetector` and a distance-based `KNNOutlierDetector` from scratch.
   - **Demo B:** Visually comparing decision boundaries of Scikit-Learn's `IsolationForest` and `OneClassSVM` on synthetic 2D data.
   - **Demo C:** Setting contamination rate thresholds to detect simulated machine failures.
   - **Summary Checklist** for Anomaly Detection.

## 1. Core Terminology & Anomaly Types

### Outlier Detection vs. Novelty Detection
- **Outlier Detection (Unsupervised):** The training data contains outliers (anomalies mixed with normal observations). The model aims to fit the central concentration of data and isolate the deviations.
- **Novelty Detection (Semi-Supervised):** The training data contains *only* normal instances. The model learns the boundary of "normalcy" so it can flag novel patterns when deployed in production.

### Types of Anomalies
1. **Point Anomalies:** A single data point deviates significantly from the rest of the dataset (e.g., a single transaction of $1,000,000$ on a card that usually spends $20).
2. **Contextual Anomalies:** The data point is anomalous *only* within a specific context (e.g., a temperature reading of $30^\circ\text{C}$ in winter is anomalous, but the same reading in summer is normal).
3. **Collective Anomalies:** A sequence or collection of data points is anomalous, even if individual points are not (e.g., a constant flat-lined heart rate reading over 5 minutes is anomalous, even though a single reading of 72 bpm is normal).

## 2. Detection Algorithms & Mathematical Foundations

### A. Statistical Approaches
- **Z-Score Method (Parametric):** Assumes the data follows a Normal distribution $N(\mu, \sigma^2)$. Points are flagged if their Z-score exceeds a threshold (typically $\pm 3$):
  $$Z_i = \frac{x_i - \mu}{\sigma}$$
- **IQR Method (Non-parametric):** Uses percentiles. Points outside the inner fence are outliers:
  $$\text{Limits} = [Q_1 - 1.5 \cdot \text{IQR}, \; Q_3 + 1.5 \cdot \text{IQR}] \quad \text{where } \text{IQR} = Q_3 - Q_1$$
- **Mahalanobis Distance:** Measures distance from a point to a multivariate distribution centroid, accounting for covariance. Useful for multi-dimensional Gaussian clusters:
  $$D_M(x) = \sqrt{(x - \mu)^T \Sigma^{-1} (x - \mu)}$$

### B. Distance & Density-Based (KNN & LOF)
- **KNN Anomaly Score:** Computes the distance of point $x$ to its $k$-nearest neighbors. Points in sparse regions will have much larger average neighbor distances than points in dense clusters.
- **Local Outlier Factor (LOF):** Measures local density deviation of a point relative to its neighbors. It compares the Local Reachability Density (LRD) of a point with those of its $k$-neighbors. If the ratio is significantly greater than 1, the point lies in a much sparser region than its neighbors, signaling an anomaly.

### C. Isolation Forest (Tree-Based)
Instead of modeling normal points, Isolation Forest explicitly **isolates** anomalies by building random recursive split trees (iTrees).
- **The Intuition:** Anomalies require fewer random splits to isolate than normal points, which are packed closely together.
- **Anomaly Score ($s$):** For a point $x$ and dataset size $n$, the score is:
  $$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$
  where $E(h(x))$ is the average path length across trees, and $c(n)$ is the average path length of an unsuccessful search in a Binary Search Tree (normalizing factor).
  - If $s \approx 1$: Path lengths are very short $\rightarrow$ easily isolated $\rightarrow$ **Anomaly**.
  - If $s \ll 0.5$: Path lengths are long $\rightarrow$ hard to isolate $\rightarrow$ **Normal**.

### D. One-Class SVM (Kernel-Based)
Maps data into a high-dimensional feature space using a kernel (like RBF) and separates the data points from the origin with a maximum-margin hyperplane.
- Points falling on the wrong side of the boundary (closest to the origin) are classified as outliers. Controlled by the hyperparameter $\nu$ (`nu`), which represents the upper bound on the fraction of training outliers.

## 3. Code Setup & Imports

Let's prepare the Python environment and import the required modules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print("Libraries imported successfully!")

## 4. Demo A: Custom Statistical & KNN Outlier Detectors From Scratch

We will build two custom scratch detectors:
1. `ZScoreOutlierDetector`: Calculates the Z-score for univariate data.
2. `KNNOutlierDetectorScratch`: Calculates multi-dimensional anomaly scores based on average nearest-neighbor distances.

In [ ]:
class ZScoreOutlierDetector:
    def __init__(self, threshold=3.0):
        self.threshold = threshold
        self.mean_ = None
        self.std_ = None

    def fit(self, X):
        self.mean_ = np.mean(X)
        self.std_ = np.std(X)
        return self

    def predict(self, X):
        # Returns -1 for anomaly, 1 for normal
        z_scores = (X - self.mean_) / (self.std_ + 1e-9)
        predictions = np.ones(len(X), dtype=int)
        predictions[np.abs(z_scores) > self.threshold] = -1
        return predictions


class KNNOutlierDetectorScratch:
    def __init__(self, k=5, contamination=0.05):
        self.k = k
        self.contamination = contamination
        self.threshold_ = None
        self.X_train_ = None

    def fit(self, X):
        self.X_train_ = X
        # Calculate pairwise distances to determine threshold on training set
        scores = self.decision_function(X)
        # Threshold is set at the percentile corresponding to contamination
        self.threshold_ = np.percentile(scores, 100 * (1 - self.contamination))
        return self

    def decision_function(self, X):
        # Compute distance to k-nearest neighbors in train set
        scores = []
        for x in X:
            # Euclidean distances to all training points
            dists = np.linalg.norm(self.X_train_ - x, axis=1)
            # Sort distances and get average of top 1 to k nearest neighbors
            # (Index 0 is self if evaluating train data, but average over k is robust)
            sorted_dists = np.sort(dists)
            k_dists = sorted_dists[1:self.k+1]
            scores.append(np.mean(k_dists))
        return np.array(scores)

    def predict(self, X):
        scores = self.decision_function(X)
        predictions = np.ones(len(X), dtype=int)
        predictions[scores > self.threshold_] = -1
        return predictions

print("Scratch anomaly detectors defined successfully!")

### Testing Scratch KNN Outlier Detector
Let's test our KNN scratch class on synthetic 2D cluster data with manual outliers injected.

In [ ]:
# Generate clean cluster data
X_normal, _ = make_blobs(n_samples=150, centers=1, cluster_std=0.8, random_state=42)

# Manually inject 8 extreme outliers
X_outliers = np.random.uniform(low=-6, high=6, size=(8, 2))
X_all = np.vstack([X_normal, X_outliers])

# Fit and predict
knn_scratch = KNNOutlierDetectorScratch(k=5, contamination=0.05)
knn_scratch.fit(X_all)
preds = knn_scratch.predict(X_all)

# Plot output
plt.figure(figsize=(9, 6))
plt.scatter(X_all[preds == 1, 0], X_all[preds == 1, 1], color='#3498db', alpha=0.7, label='Normal (Class 1)', edgecolors='black')
plt.scatter(X_all[preds == -1, 0], X_all[preds == -1, 1], color='#e74c3c', marker='x', s=80, fontweight='bold', label='Outlier (Class -1)')
plt.title("Scratch KNN Distance-Based Outlier Detector", fontsize=13, fontweight='bold')
plt.legend()
plt.show()

## 5. Demo B: Visually Comparing Isolation Forest vs. One-Class SVM

Now we'll generate two separate clusters of normal data and compare how Scikit-Learn's `IsolationForest` and `OneClassSVM` model the decision boundaries in 2D space.

In [ ]:
# Generate dual cluster normal data + noise
X_normal2, _ = make_blobs(n_samples=300, centers=[[-3, -3], [3, 3]], cluster_std=1.0, random_state=42)
X_noise = np.random.uniform(low=-8, high=8, size=(25, 2))
X_data = np.vstack([X_normal2, X_noise])

# Fit models
iso_forest = IsolationForest(contamination=0.08, random_state=42)
preds_if = iso_forest.fit_predict(X_data)

oc_svm = OneClassSVM(kernel='rbf', gamma=0.1, nu=0.08)
preds_svm = oc_svm.fit_predict(X_data)

# Mesh grid for contour boundary visualization
xx, yy = np.meshgrid(np.arange(-9, 9, 0.05), np.arange(-9, 9, 0.05))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

Z_if = iso_forest.decision_function(grid_pts).reshape(xx.shape)
Z_svm = oc_svm.decision_function(grid_pts).reshape(xx.shape)

# Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Isolation Forest Plot
axes[0].contourf(xx, yy, Z_if, cmap='coolwarm', alpha=0.2)
axes[0].scatter(X_data[preds_if == 1, 0], X_data[preds_if == 1, 1], color='#3498db', alpha=0.6, label='Normal')
axes[0].scatter(X_data[preds_if == -1, 0], X_data[preds_if == -1, 1], color='red', marker='x', s=50, label='Anomaly')
axes[0].set_title("Isolation Forest Boundaries\n(Isolates points via random splits)", fontsize=13, fontweight='bold')
axes[0].legend()

# One-Class SVM Plot
axes[1].contourf(xx, yy, Z_svm, cmap='coolwarm', alpha=0.2)
axes[1].scatter(X_data[preds_svm == 1, 0], X_data[preds_svm == 1, 1], color='#3498db', alpha=0.6, label='Normal')
axes[1].scatter(X_data[preds_svm == -1, 0], X_data[preds_svm == -1, 1], color='red', marker='x', s=50, label='Anomaly')
axes[1].set_title("One-Class SVM Boundaries\n(Hyperplane envelope around dense clusters)", fontsize=13, fontweight='bold')
axes[1].legend()

plt.suptitle("Anomaly Detection Model Boundary Visualizations", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 6. Demo C: Industrial Equipment Failure Prediction (Evaluation)

We simulate sensor readings of industrial equipment. Normal state has temperatures around $70^\circ\text{C}$ and pressure around $50$ psi. Rare sensor spikes (anomalies) occur when failures are imminent. We will check model performance metrics (Precision and Recall).

In [ ]:
# Simulate 1000 normal operations
normal_temp = np.random.normal(loc=70, scale=3, size=950)
normal_pressure = np.random.normal(loc=50, scale=4, size=950)

# Simulate 50 equipment failures (spikes)
failure_temp = np.random.uniform(low=85, high=100, size=50)
failure_pressure = np.random.uniform(low=70, high=90, size=50)

df_normal = pd.DataFrame({'temp': normal_temp, 'pressure': normal_pressure, 'is_anomaly': 0})
df_failures = pd.DataFrame({'temp': failure_temp, 'pressure': failure_pressure, 'is_anomaly': 1})
df_sensor = pd.concat([df_normal, df_failures], ignore_index=True)

X_features = df_sensor[['temp', 'pressure']].values
y_true = df_sensor['is_anomaly'].values  # 1 for failure, 0 for normal

# Train Isolation Forest (assume we estimate around 5% failure contamination rate)
model_if = IsolationForest(contamination=0.05, random_state=42)
preds_raw = model_if.fit_predict(X_features)

# Convert output: Isolation Forest returns -1 for anomaly, 1 for normal.
# We map -1 -> 1 (Anomaly) and 1 -> 0 (Normal) to match ground truth.
y_pred = np.zeros(len(preds_raw), dtype=int)
y_pred[preds_raw == -1] = 1

# Calculate evaluation metrics
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Normal Operating', 'Equipment Failure']))

## Summary Checklist for Anomaly Detection

1. **Confirm Your Setting (Outlier vs. Novelty):**
   - If your training data contains corrupt entries, use unsupervised *Outlier Detection* (e.g., `IsolationForest`).
   - If your training data is clean and you want to detect new patterns, train a semi-supervised *Novelty Detector* (e.g., `OneClassSVM` fit on normal-only data).
2. **Algorithm Strengths:**
   - **Isolation Forest:** Gold standard for tabular data, handles high dimensionality well, fast execution times.
   - **One-Class SVM:** Excellent at handling complex non-linear boundaries via kernels, but sensitive to hyperparameter tuning (`gamma`, `nu`).
   - **LOF:** Best for identifying local outliers that might be missed by global thresholds, but computationally heavy on massive datasets.
3. **Tuning Contamination Rate:**
   - The `contamination` parameter is the expected proportion of outliers in the data. Setting this accurately is vital because it determines the threshold score boundary.